# 06: Measure your circuit on a quantum computer

**What you will learn:** ideal probabilities describe a model; real hardware returns finite-shot counts. A shot is one execution followed by measurement.

We will send a three-qubit asymmetric RY circuit to Quafu and compare its counts with the ideal prediction.

**Learning goal:** submit an asymmetric circuit, read shot counts and compare them with ideal probabilities in a known bit order.

**Prerequisites:** lesson 01; lesson 05 is helpful for the job lifecycle. Real execution requires Quafu credentials, an available target and shot quota. Local rehearsal does not exercise hardware.

Choose a route in the [workshop guide](../README.md).


## 1. Prepare the workshop

Run setup unchanged. Platform credentials are read automatically, never entered or displayed here. This lesson uses independent asymmetric-result files; it does not overwrite or reuse old Bell receipts.


In [ ]:
from pathlib import Path
import os
import sys
import json

search_roots = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
WORKSHOP = next((root / "flagos2026" for root in search_roots
                 if (root / "flagos2026/notebooks").is_dir()), None)
if WORKSHOP is None:
    raise RuntimeError("Cannot locate flagos2026/notebooks. Open the deployed workshop first.")
sys.path.insert(0, str(WORKSHOP / "scripts"))
from cloud_lesson import workshop_directory

WORKSHOP = workshop_directory(Path.cwd())
repository = WORKSHOP.parent.parent
ROOT = repository if (repository / "pyproject.toml").is_file() else WORKSHOP
OUTPUTS = WORKSHOP / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)


## 2. Build the circuit and check the ideal answer

Apply RY(pi/3) only to q0. The ideal probabilities are P(000)=0.75 and P(100)=0.25; q1 and q2 stay zero. This asymmetric distribution exposes a bit-order reversal.

Each horizontal line in the circuit diagram is a qubit; gates execute from left to right. Vertical wire order is not the bit-string display order on a cloud platform.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

import math
import flagquantum as fq

circuit = fq.Circuit(3).ry(0, math.pi / 3)
fig, ax = circuit.draw(format="mpl", show_all_qubits=True)
plt.show()
ideal = fq.run(circuit)
labels = [format(index, "03b") for index in range(8)]
ideal_probabilities = ideal.to_statevector().reshape(-1).abs().square().tolist()
print("Ideal probabilities (q0 q1 q2):", dict(zip(labels, ideal_probabilities)))


## 3. Choose the hardware

Confirm today’s available target with your instructor. A shot is one execution and measurement. **Enabling the switch and running the submission cell creates a real QPU task with 1024 shots and uses your quota.**

Published examples default to local rehearsal (`False`). For the event deployment, set the visible switch to `True` after the instructor verifies credentials, target and quota, so sequential execution really submits. The actual target, shots and result directory are printed below; detecting credentials alone never enables submission.


In [ ]:
target = os.getenv("QUAFU_TARGET", "quafu:Baihua")
shots = 1024
submit_to_hardware = False

print("Target:", target)
print("Shots:", shots)
print("Result directory:", OUTPUTS)
print("Real submission enabled:", submit_to_hardware)


### Choose the receipt location (run unchanged)

This cell selects a file in your personal persistent result directory and checks for uncertain attempts from earlier workshop versions. The SDK handles saving, configuration checks and matching-request recovery through `receipt=`. Keep the receipt and its SDK-created journal together.

Existing manual receipts need explicit `fq.restore_job(receipt_path)`; they are not silently adopted. Never delete old records to force another submission.


In [ ]:
receipt_path = OUTPUTS / "quafu-ry-q0-three-qubits.receipt.json"
job = None
if submit_to_hardware:
    if not os.getenv("QUAFU_API_KEY"):
        raise RuntimeError("Configure Quafu access outside this notebook; nothing submitted.")
    previous_files = (
        OUTPUTS / "quafu-ry-q0-three-qubits.submission.json",
        OUTPUTS / "quafu-ry-q0-three-qubits.json",
    )
    if not receipt_path.exists() and any(path.exists() for path in previous_files):
        raise RuntimeError("Earlier workshop records exist without a receipt. Keep them and reconcile the task; do not resubmit.")
print("Receipt path:", receipt_path.resolve())


## 4. Submit through FlagQuantum

`fq.submit(..., receipt=receipt_path)` submits and saves on the first run, then restores on matching reruns without submitting again. It returns without waiting for execution. Configuration changes, incomplete records and recovery failures raise instead of creating another task; no manual hashing or `job.save()` is needed. A new experiment needs a new receipt path after checking the old attempt.


In [ ]:
job = None
if submit_to_hardware:
    job = fq.submit(
        circuit, target=target, shots=shots, outputs=fq.counts(), compiler=None,
        receipt=receipt_path,
    )
    print("Task ID:", job.id)
else:
    print("Local rehearsal only. Enable the switch to submit a real task.")


## 5. Query progress

This is a single read-only query. `queued` means waiting, `running` means executing, and `succeeded` means the platform reports completion. Result delivery can still be delayed or unavailable.

**Read the output:** `Task ID` identifies the task, not a completed calculation. A matching rerun prints the same ID because the SDK restored it. Retrieve the result before comparing local and remote execution.

**Rerun safely:** repeat the same submission call with the same persistent receipt path; the SDK validates and restores it. To check a queued task, rerun only the result cell. Never delete recovery files to retry an uncertain submission. Changed circuit, target or shots needs a new experiment path after checking the old attempt.


In [ ]:
if job is not None:
    print("Status:", job.status())


## 6. Wait briefly and read the result

`job.wait()` is a convenience for polling status and retrieving the result. A timeout leaves the task running. Rerun this cell later. Alternatively use `job.result()` once the status is `succeeded`. No local result is substituted for a remote failure.

**Concept:** ideal probabilities describe the noiseless model; counts are observations from a finite number of shots. Here P(000)=0.75 and P(100)=0.25, so 1024 ideal shots have expected counts 768 and 256, not guaranteed exact counts.

**Expected output:** the comparison below shows ideal probability, observed count and observed frequency for every bit string. FlagQuantum returns full-register strings in q0,q1,q2 order. In the previously verified Quafu platform display with identity measurement mapping q0→c0, q1→c1, q2→c2, strings were c2,c1,c0: platform 001 corresponds to FlagQuantum 100. For reordered or partial measurements, inspect the actual measurement mapping rather than blindly reversing strings. FlagQuantum already normalizes returned counts; do not reverse them again.

**Common misconception:** finite-shot sampling causes random frequency fluctuations even for an ideal device. Device noise can shift probabilities and populate ideally impossible outcomes. More shots reduce sampling uncertainty, not systematic device errors. A single histogram cannot uniquely identify each noise mechanism.

**Example of reading a row (illustration, not a hardware result):** `100 ideal: 0.25 count: 256 frequency: 0.25` means 256 of 1024 measurements returned q0=1,q1=0,q2=0. Real counts need not equal 256. Both bars use probability/frequency units; raw counts are printed, not plotted on that scale.


In [ ]:
result = None
if job is not None:
    try:
        result = job.wait(timeout=30)
    except TimeoutError:
        print("Result not available yet; last provider status:", job.raw_status)
        print("Rerun only this result cell later; do not resubmit. If completion is reported but results remain unavailable, ask the instructor to inspect task logs.")
if result is not None:
    counts = result.counts[0]
    assert sum(counts.values()) == shots
    print("Measured counts:", counts)
    print("Total shots:", sum(counts.values()))

    frequencies = [counts.get(label, 0) / shots for label in labels]
    for label, probability, frequency in zip(labels, ideal_probabilities, frequencies):
        print(label, "ideal:", round(probability, 4), "count:", counts.get(label, 0), "frequency:", round(frequency, 4))
    positions = list(range(len(labels)))
    plt.bar([p - 0.2 for p in positions], ideal_probabilities, width=0.4, label="Ideal probability")
    plt.bar([p + 0.2 for p in positions], frequencies, width=0.4, label="Hardware frequency")
    plt.xticks(positions, labels)
    plt.xlabel("Bit string (q0 q1 q2)")
    plt.ylabel("Probability / frequency")
    plt.legend()
    plt.show()


### Save this asymmetric experiment's result (optional)

This folded export records the actual task ID and counts. It is not required to understand or use `fq.submit()`.


In [ ]:
if result is not None:
    record = {
        "source": "live", "status": "completed", "target": target,
        "shots": shots, "task_id": job.id, "counts": counts,
    }
    (OUTPUTS / "quafu-ry-q0-three-qubits.json").write_text(json.dumps(record, indent=2))


## Take it home

The standard API is the same outside Jupyter. The first call submits real work and saves its receipt; matching reruns with the same persistent path restore it. The SDK owns configuration checks and duplicate-submission guards. Configure access outside your source first.

```python
import math
import flagquantum as fq

circuit = fq.Circuit(3).ry(0, math.pi / 3)
job = fq.submit(
    circuit, target="quafu:Baihua", shots=1024, outputs=fq.counts(),
    receipt="ry-q0-three-qubits.receipt.json",
)
print(job.id, job.status())
```

For a saved task: `job = fq.restore_job("ry-q0-three-qubits.receipt.json")`. When complete, inspect `job.result().counts`.

**Explain the result:** why might `001` or `110` appear? Would more shots remove device noise? Old Bell results belong to another experiment; do not compare them against this ideal distribution. See [Cloud job field notes](../CLOUD_JOBS.md) for setup and recovery details.
